In [8]:
import platform
import netket as nk

print("Python version (requires >=3.9)", platform.python_version())
print("NetKet version (requires >=3.9.1)", nk.__version__)

Python version (requires >=3.9) 3.11.8
NetKet version (requires >=3.9.1) 3.18


In [13]:
# Define the system
L = 4
g = nk.graph.Hypercube(length=L, n_dim=2, pbc=True)
hi = nk.hilbert.Spin(s=1 / 2, N=g.n_nodes)

# Build the Hamiltonian
hamiltonian = nk.operator.LocalOperator(hi)

# Add transverse field terms
for site in g.nodes():
    hamiltonian = hamiltonian - 1.0 * nk.operator.spin.sigmax(hi, site)

# Add Ising interaction terms
for i, j in g.edges():
    hamiltonian = hamiltonian + nk.operator.spin.sigmaz(
        hi, i
    ) @ nk.operator.spin.sigmaz(hi, j)

# Convert to JAX format
hamiltonian_jax = hamiltonian.to_pauli_strings().to_jax_operator()

# Compute exact ground state for comparison
from scipy.sparse.linalg import eigsh

e_gs, psi_gs = eigsh(hamiltonian.to_sparse(), k=1)
e_gs = e_gs[0]
psi_gs = psi_gs.reshape(-1)

print(f"Exact ground state energy: {e_gs:.6f}")

Exact ground state energy: 34.010598


In [11]:
import flax.linen as nn
import jax
import jax.numpy as jnp
import numpy as np
from flax.nnx import to_linen
# A Flax model must be a class subclassing `nn.Module`
class MF(nn.Module):
    @nn.compact
    def __call__(self, x):
        # - The dtype of the tensor.
        lam = self.param("lambda", nn.initializers.normal(), (1,), float)
        # compute the probabilities
        p = nn.log_sigmoid(lam * x)
        # sum the output
        return 0.5 * jnp.sum(p, axis=-1)
model = MF()
parameters = model.init(jax.random.key(0), np.ones((hi.size,)))

ImportError: cannot import name 'to_linen' from 'flax.nnx' (/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/flax/nnx/__init__.py)

In [12]:
import jax
import jax.numpy as jnp
from flax import nnx

class Linear(nnx.Module):
    def __init__(self, din: int, dout: int, *, rngs: nnx.Rngs):
        key = rngs.params()
        self.w = nnx.Param(jax.random.uniform(key, (din, dout)))
        self.b = nnx.Param(jnp.zeros((dout,)))
        self.din, self.dout = din, dout

    def __call__(self, x: jax.Array):
        return x @ self.w + self.b

# 创建模型（参数已经初始化）
model = Linear(2, 5, rngs=nnx.Rngs(params=0))

# ✅ 提取参数的正确方法
graphdef, parameters = nnx.split(model, nnx.Param)

# 传递给 sampler
sampler_state = sampler.init_state(model, parameters, seed=1)


NameError: name 'sampler' is not defined